# Understanding MCP and A2A: Building Intelligent Agent Systems with Granite

As enterprise AI systems evolve beyond single-agent architectures, two protocols have emerged as foundational standards: **Model Context Protocol (MCP)** for tool integration and **Agent-to-Agent (A2A)** for multi-agent orchestration. This guide demonstrates both protocols using IBM Granite models, providing practical insights for architects designing production agent systems.

## What You'll Learn

1. **MCP fundamentals**: How agents interact with tools and data sources through standardized interfaces
2. **A2A fundamentals**: How agents collaborate as autonomous peers in distributed systems
3. **Practical implementation**: Working code examples using Granite models
4. **Architectural guidance**: When to use MCP, A2A, or both in your systems

We'll explore these concepts through a travel planning scenario, implementing the same use case with both protocols to highlight their distinct characteristics.

## Prerequisites

This notebook assumes:

- Python 3.8+ with basic async/await knowledge
- Familiarity with agent frameworks (LangChain, LangGraph, or similar)
- Access to Granite models via Ollama or watsonx.ai
- Understanding of JSON Schema and REST APIs

### Environment Setup

Ensure you have Ollama running locally with a Granite model:

```bash
ollama serve
ollama pull granite3-dense:8b
```

In [ ]:
# Install required dependencies
# %pip install langchain langgraph langchain_ibm httpx anyio

## Conceptual Framework: MCP vs A2A

### Model Context Protocol (MCP)

**MCP standardizes how agents access tools and data sources.** Think of it as a universal adapter that allows any agent to invoke capabilities—databases, APIs, file systems, or computational services—through a consistent interface.

**Key characteristics:**
- **Direction**: Agent → Tool/Data
- **Abstraction**: Individual functions or capabilities
- **Interface**: JSON Schema-based tool descriptions
- **Use case**: Exposing internal capabilities to agents

### Agent-to-Agent Protocol (A2A)

**A2A standardizes how agents collaborate as peers.** Rather than exposing low-level tools, agents publish high-level capabilities through Agent Cards and communicate via JSON-RPC 2.0 messages.

**Key characteristics:**
- **Direction**: Agent → Agent
- **Abstraction**: Complete agent capabilities (skills)
- **Interface**: Agent Cards with skill definitions
- **Use case**: Multi-agent systems with distributed ownership

### Architectural Layers

1. **Data & Tools Layer**: Databases, APIs, services (accessed via MCP)
2. **Agent Layer**: Granite-powered agents with reasoning capabilities
3. **Agent Network Layer**: Multi-agent collaboration (coordinated via A2A)

MCP operates between layers 1 and 2, while A2A operates within layer 3.

## Granite Model Integration

Both protocols leverage Granite models for reasoning and natural language generation.

In [ ]:
import os
import httpx

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
GRANITE_MODEL_ID = os.environ.get("GRANITE_MODEL_ID", "granite3-dense:8b")


async def call_granite(system_prompt: str, user_prompt: str) -> str:
    """Invoke Granite model via Ollama's HTTP API."""
    payload = {
        "model": GRANITE_MODEL_ID,
        "prompt": f"<system>{system_prompt}</system>\n<user>{user_prompt}</user>",
        "stream": False,
    }
    
    async with httpx.AsyncClient(base_url=OLLAMA_BASE_URL, timeout=60.0) as client:
        resp = await client.post("/api/generate", json=payload)
        resp.raise_for_status()
        data = resp.json()
        return data.get("response", "")


print(f"Granite model configured: {GRANITE_MODEL_ID}")

## MCP Implementation: Travel Planning with Tools

We'll expose three tools—flight search, hotel booking, and weather forecasting—that a single Granite-powered agent orchestrates.

### Tool Definitions

Each MCP tool requires:
- **Name**: Unique identifier
- **Description**: Natural language explanation
- **Input Schema**: JSON Schema defining parameters
- **Handler**: Implementation logic

In [ ]:
import asyncio
from dataclasses import dataclass
from typing import Dict, Any


@dataclass
class MCPToolSpec:
    """MCP tool specification."""
    name: str
    description: str
    input_schema: Dict[str, Any]


# Mock data
MCP_MOCK_FLIGHTS = {
    "Albany": {"date": "2026-05-21", "time": "10:00 AM", "flight": "UA321", "price": 285},
}

MCP_MOCK_HOTELS = {
    "Albany": {"check_in": "2026-05-21", "nights": 2, "hotel": "Albany Grand Hotel", "price": 195},
}

MCP_MOCK_WEATHER = {
    "Albany": {"date": "2026-05-21", "forecast": "Partly cloudy, 22°C with light breeze"},
}


# Tool specifications
find_flight_spec = MCPToolSpec(
    name="find_flight",
    description="Search for available flights to a destination.",
    input_schema={
        "type": "object",
        "properties": {
            "destination": {"type": "string"},
            "date": {"type": "string"},
        },
        "required": ["destination", "date"],
    },
)

book_hotel_spec = MCPToolSpec(
    name="book_hotel",
    description="Book hotel accommodation.",
    input_schema={
        "type": "object",
        "properties": {
            "destination": {"type": "string"},
            "check_in": {"type": "string"},
            "nights": {"type": "integer"},
            "flight_arrival": {"type": "string"},
        },
        "required": ["destination", "check_in", "nights", "flight_arrival"],
    },
)

get_forecast_spec = MCPToolSpec(
    name="get_forecast",
    description="Retrieve weather forecast.",
    input_schema={
        "type": "object",
        "properties": {
            "destination": {"type": "string"},
            "date": {"type": "string"},
        },
        "required": ["destination", "date"],
    },
)


def invoke_mcp_travel_tool(tool: MCPToolSpec, args: Dict[str, Any]) -> Dict[str, Any]:
    """MCP tool dispatcher."""
    if tool.name == "find_flight":
        return {"flight": MCP_MOCK_FLIGHTS.get(args["destination"], {})}
    if tool.name == "book_hotel":
        return {"hotel": MCP_MOCK_HOTELS.get(args["destination"], {})}
    if tool.name == "get_forecast":
        return {"weather": MCP_MOCK_WEATHER.get(args["destination"], {})}
    raise ValueError(f"Unknown tool: {tool.name}")


print("MCP tools configured")

### MCP Orchestration

The orchestrator:
1. Calls tools sequentially
2. Aggregates structured responses
3. Uses Granite for natural language synthesis

In [ ]:
async def plan_trip_with_mcp_tools(destination: str, date: str) -> str:
    """Travel planning using MCP tools."""
    flight_resp = invoke_mcp_travel_tool(find_flight_spec, {"destination": destination, "date": date})
    flight = flight_resp.get("flight")

    hotel_resp = invoke_mcp_travel_tool(
        book_hotel_spec,
        {"destination": destination, "check_in": date, "nights": 2, "flight_arrival": flight["time"] if flight else "unknown"},
    )
    hotel = hotel_resp.get("hotel")

    weather_resp = invoke_mcp_travel_tool(get_forecast_spec, {"destination": destination, "date": date})
    weather = weather_resp.get("weather")

    system_prompt = "You are an expert travel advisor. Create a concise travel plan."
    user_prompt = f"""Destination: {destination}
Date: {date}
Flight: {flight}
Hotel: {hotel}
Weather: {weather}

Generate a 2-day travel plan."""

    return await call_granite(system_prompt, user_prompt)


async def demo_mcp_travel():
    print("\n=== MCP Travel Planning ===")
    plan = await plan_trip_with_mcp_tools("Albany", "2026-05-21")
    print(plan)


await demo_mcp_travel()

## A2A Implementation: Multi-Agent Travel Planning

Now we'll create specialized agents that communicate as peers:
- **Client Agent**: Orchestrates workflow
- **Flight Agent**: Handles flight operations
- **Hotel Agent**: Manages accommodations
- **Weather Agent**: Provides forecasts

In [ ]:
from dataclasses import dataclass, field

# Mock data (consistent with MCP)
MOCK_FLIGHTS = {"Albany": {"date": "2026-05-21", "time": "10:00 AM", "flight": "UA321", "price": 285}}
MOCK_HOTELS = {"Albany": {"check_in": "2026-05-21", "nights": 2, "hotel": "Albany Grand Hotel", "price": 195}}
MOCK_WEATHER = {"Albany": {"date": "2026-05-21", "forecast": "Partly cloudy, 22°C with light breeze"}}


@dataclass
class AgentCard:
    """A2A Agent Card."""
    name: str
    description: str
    skills: Dict[str, Dict[str, Any]] = field(default_factory=dict)


@dataclass
class A2AMessage:
    """A2A JSON-RPC message."""
    sender: str
    receiver: str
    method: str
    params: Dict[str, Any]
    id: str


class FlightAgent:
    def __init__(self):
        self.card = AgentCard(
            name="flight_agent",
            description="Flight search specialist.",
            skills={"find_flight": {"description": "Search flights"}},
        )

    async def handle(self, message: A2AMessage) -> Dict[str, Any]:
        if message.method != "find_flight":
            raise ValueError(f"Unsupported method: {message.method}")
        return {"flight": MOCK_FLIGHTS.get(message.params["destination"], {})}


class HotelAgent:
    def __init__(self):
        self.card = AgentCard(
            name="hotel_agent",
            description="Hotel booking specialist.",
            skills={"book_hotel": {"description": "Book hotels"}},
        )

    async def handle(self, message: A2AMessage) -> Dict[str, Any]:
        if message.method != "book_hotel":
            raise ValueError(f"Unsupported method: {message.method}")
        return {"hotel": MOCK_HOTELS.get(message.params["destination"], {})}


class WeatherAgent:
    def __init__(self):
        self.card = AgentCard(
            name="weather_agent",
            description="Weather forecast specialist.",
            skills={"get_forecast": {"description": "Get forecasts"}},
        )

    async def handle(self, message: A2AMessage) -> Dict[str, Any]:
        if message.method != "get_forecast":
            raise ValueError(f"Unsupported method: {message.method}")
        return {"weather": MOCK_WEATHER.get(message.params["destination"], {})}


print("A2A agents initialized")

### A2A Orchestration

The Client Agent:
1. Discovers agents via Agent Cards
2. Sends JSON-RPC messages
3. Aggregates responses
4. Uses Granite for synthesis

In [ ]:
class ClientAgent:
    def __init__(self, flight_agent, hotel_agent, weather_agent):
        self.flight_agent = flight_agent
        self.hotel_agent = hotel_agent
        self.weather_agent = weather_agent

    async def plan_trip(self, destination: str, date: str) -> str:
        """Orchestrate multi-agent travel planning."""
        flight_msg = A2AMessage("client_agent", "flight_agent", "find_flight", {"destination": destination, "date": date}, "req-1")
        flight_resp = await self.flight_agent.handle(flight_msg)
        flight = flight_resp.get("flight")

        hotel_msg = A2AMessage(
            "client_agent", "hotel_agent", "book_hotel",
            {"destination": destination, "check_in": date, "nights": 2, "flight_arrival": flight["time"] if flight else "unknown"},
            "req-2"
        )
        hotel_resp = await self.hotel_agent.handle(hotel_msg)
        hotel = hotel_resp.get("hotel")

        weather_msg = A2AMessage("client_agent", "weather_agent", "get_forecast", {"destination": destination, "date": date}, "req-3")
        weather_resp = await self.weather_agent.handle(weather_msg)
        weather = weather_resp.get("weather")

        system_prompt = "You are an expert travel advisor. Create a concise travel plan."
        user_prompt = f"""Destination: {destination}
Date: {date}
Flight: {flight}
Hotel: {hotel}
Weather: {weather}

Generate a 2-day travel plan."""

        return await call_granite(system_prompt, user_prompt)


client_agent = ClientAgent(FlightAgent(), HotelAgent(), WeatherAgent())
print("Client agent initialized")

In [ ]:
async def demo_a2a_travel():
    print("\n=== A2A Travel Planning ===")
    plan = await client_agent.plan_trip("Albany", "2026-05-21")
    print(plan)


await demo_a2a_travel()

## Comparative Analysis

| Aspect | MCP | A2A |
|--------|-----|-----|
| **Purpose** | Agent-to-tool interactions | Agent-to-agent collaboration |
| **Direction** | Agent → Tool | Agent → Agent |
| **Abstraction** | Functions | Complete agents |
| **Interface** | Tool schemas | Agent Cards |
| **Messages** | Tool invocations | JSON-RPC 2.0 |
| **Ownership** | Centralized | Distributed |
| **Use Case** | Internal tools | Multi-agent systems |

### Decision Framework

**Choose MCP when:**
- Single team owns agents and tools
- Need standardized tool access
- Building within one runtime

**Choose A2A when:**
- Multiple teams own different agents
- Agents run in different systems
- Need stable agent-level contracts

**Use both when:**
- Building enterprise multi-agent systems
- MCP for internal tools, A2A for agent communication

## Common Questions

**Q: Is A2A just another way to call tools?**

A: No. A2A treats entire agents as autonomous peers with reasoning capabilities. Agents may use tools internally (via MCP), but A2A focuses on agent-level collaboration.

**Q: Can I use MCP without A2A?**

A: Yes. Many single-agent systems only need MCP for tool access.

**Q: Can I use A2A without MCP?**

A: Yes. A2A doesn't require MCP, though MCP provides a natural approach for internal tool access.

**Q: Where does Granite fit?**

A: Granite provides the reasoning engine. MCP and A2A standardize *how* Granite-powered components interact, not *what* they do.

**Q: What about performance?**

A: MCP typically has lower latency (local tools). A2A involves network communication but enables distribution and scale.

## Summary

**Model Context Protocol (MCP)** standardizes agent-to-tool interactions, enabling:
- Consistent tool interfaces
- Reusable implementations
- Centralized security

**Agent-to-Agent Protocol (A2A)** standardizes agent collaboration, enabling:
- Distributed agent networks
- Independent evolution
- Cross-team collaboration

**Key Takeaways:**
1. Use MCP for tool integration, A2A for agent collaboration
2. Granite powers reasoning in both protocols
3. Production systems often use both together
4. Choose based on ownership, scale, and architecture

### Resources
- [Granite Agent Cookbook](https://github.com/ibm-granite-community/granite-agent-cookbook)
- [MCP Specification](https://modelcontextprotocol.io/)
- [A2A Protocol](https://a2a-protocol.org/latest/)